In [0]:
%sql
-- =====================================================
-- KPI 1 : Vue d'ensemble du taux de fraude
-- Usage métier : Dashboard direction risque
-- =====================================================

SELECT 
    transaction_type,
    COUNT(*) AS nb_transactions,
    ROUND(SUM(Amount), 2) AS montant_total,
    ROUND(AVG(Amount), 2) AS montant_moyen,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pourcentage
FROM banking_lakehouse.gold.fact_transactions
GROUP BY transaction_type
ORDER BY nb_transactions DESC;

In [0]:
%sql
-- =====================================================
-- KPI 2 : Distribution des fraudes par heure de la journée
-- Usage métier : Détecter des patterns temporels suspects
-- =====================================================

SELECT 
    hour_of_day,
    COUNT(*) AS nb_total_transactions,
    SUM(CASE WHEN transaction_type = 'Fraud' THEN 1 ELSE 0 END) AS nb_fraudes,
    ROUND(
        100.0 * SUM(CASE WHEN transaction_type = 'Fraud' THEN 1 ELSE 0 END) / COUNT(*), 
        4
    ) AS taux_fraude_pct
FROM banking_lakehouse.gold.fact_transactions
GROUP BY hour_of_day
ORDER BY taux_fraude_pct DESC
LIMIT 10;

In [0]:
%sql
-- =====================================================
-- KPI 3 : Analyse du Churn par segment géographique
-- Usage métier : Dashboard rétention client
-- Note : on filtre sur is_current=True (état actuel uniquement)
-- =====================================================

SELECT 
    Geography,
    COUNT(*) AS nb_clients,
    SUM(Exited) AS nb_clients_churned,
    ROUND(100.0 * SUM(Exited) / COUNT(*), 2) AS taux_churn_pct,
    ROUND(AVG(CreditScore), 0) AS credit_score_moyen,
    ROUND(AVG(Balance), 2) AS balance_moyenne
FROM banking_lakehouse.gold.dim_client
WHERE is_current = true
GROUP BY Geography
ORDER BY taux_churn_pct DESC;

In [0]:
%sql
-- =====================================================
-- Window Functions : Running Total quotidien + Ranking
-- Usage métier : Suivi de tendance et identification des
-- pics d'activité frauduleuse
-- =====================================================

WITH daily_stats AS (
    SELECT 
        transaction_date,
        COUNT(*) AS nb_transactions_jour,
        SUM(CASE WHEN transaction_type = 'Fraud' THEN 1 ELSE 0 END) AS nb_fraudes_jour,
        SUM(Amount) AS montant_total_jour
    FROM banking_lakehouse.gold.fact_transactions
    GROUP BY transaction_date
)

SELECT 
    transaction_date,
    nb_transactions_jour,
    nb_fraudes_jour,
    montant_total_jour,
    -- Running Total (cumul depuis le début)
    SUM(nb_transactions_jour) OVER (ORDER BY transaction_date) AS cumul_transactions,
    -- Rang du jour selon le nombre de fraudes (identifier les pics)
    RANK() OVER (ORDER BY nb_fraudes_jour DESC) AS rang_jour_plus_fraude,
    -- Moyenne mobile sur les jours précédents (tendance)
    ROUND(AVG(nb_fraudes_jour) OVER (ORDER BY transaction_date ROWS BETWEEN 1 PRECEDING AND CURRENT ROW), 2) AS moyenne_mobile_fraudes
FROM daily_stats
ORDER BY transaction_date;